In [1]:
import pandas as pd

In [ ]:
system_status_mapping = {
    "TD": "Tropical Depression (<34 knots)",
    "TS": "Tropical Storm (34-63 knots)",
    "HU": "Hurricane (>64 knots)",
    "EX": "Extratropical Cyclone",
    "SD": "Subtropical Depression (<34 knots)",
    "SS": "Subtropical Storm (>34 knots)",
    "LO": "Low (not tropical/subtropical/extratropical)",
    "WV": "Tropical Wave",
    "DB": "Disturbance"
}

def clean_lat(lat_str):
    lat_str = lat_str.strip()
    if lat_str.endswith('N'):
        return float(lat_str[:-1])
    elif lat_str.endswith('S'):
        return -float(lat_str[:-1])
    else:
        return float(lat_str)

def clean_lon(lon_str):
    lon_str = lon_str.strip()
    if lon_str.endswith('E'):
        return float(lon_str[:-1])
    elif lon_str.endswith('W'):
        return -float(lon_str[:-1])
    else:
        return float(lon_str)

In [ ]:
# Parse HURDAT2 file from 2020
storm_entries = []
with open("hurdat2-2020-2024-040425.txt", "r") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    line = lines[i].strip()
    if not line or ',' not in line:
        i += 1
        continue
    # Header line: e.g. AL092021, IDA, 40,...
    parts = line.split(',')
    if len(parts) < 4:
        i += 1
        continue
    storm_id = parts[0].strip()
    storm_name = parts[1].strip()
    n_entries = int(parts[2].strip())
    year = storm_id[4:8]
    # Next n_entries lines are the storm track
    for j in range(i+1, i+1+n_entries):
        entry = lines[j].strip().split(',')
        if len(entry) < 8:
            continue
        date = entry[0].strip()  # YYYYMMDD
        time = entry[1].strip()  # HHMM
        record_identifier = entry[2].strip()
        system_status = entry[3].strip()
        lat = entry[4].strip()
        lon = entry[5].strip()
        max_sustained_wind = entry[6].strip()
        # Add all columns to dict
        storm_entries.append({
            "storm_id": storm_id,
            "storm_name": storm_name,
            "year": year,
            "date": date,
            "time": time,
            "record_identifier": record_identifier,
            "system_status": system_status,
            "lat": lat,
            "lon": lon,
            "max_sustained_wind": max_sustained_wind
        })
    i += n_entries + 1

storms_df_new = pd.DataFrame(storm_entries)

storms_df_new["datetime"] = pd.to_datetime(storms_df_new["date"] + " " + storms_df_new["time"], format="%Y%m%d %H%M")
storms_df_new = storms_df_new.drop(columns=["year", "date", "time", "record_identifier"])
storms_df_new['lat'] = storms_df_new['lat'].apply(clean_lat)
storms_df_new['lon'] = storms_df_new['lon'].apply(clean_lon)
storms_df_new["system_status_desc"] = storms_df_new["system_status"].map(system_status_mapping)

storms_df_new.to_pickle("hurdat2_storm_data_2020_2025.pkl")
storms_df_new.to_csv("hurdat2_storm_data_2020_2025.csv", index=False)

In [ ]:
# Parse HURDAT2 file from 1851
storm_entries = []
with open("hurdat2-1851-2024-040425.txt", "r") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    line = lines[i].strip()
    if not line or ',' not in line:
        i += 1
        continue
    # Header line: e.g. AL092021, IDA, 40,...
    parts = line.split(',')
    if len(parts) < 4:
        i += 1
        continue
    storm_id = parts[0].strip()
    storm_name = parts[1].strip()
    n_entries = int(parts[2].strip())
    year = storm_id[4:8]
    # Next n_entries lines are the storm track
    for j in range(i+1, i+1+n_entries):
        entry = lines[j].strip().split(',')
        if len(entry) < 8:
            continue
        date = entry[0].strip()  # YYYYMMDD
        time = entry[1].strip()  # HHMM
        record_identifier = entry[2].strip()
        system_status = entry[3].strip()
        lat = entry[4].strip()
        lon = entry[5].strip()
        max_sustained_wind = entry[6].strip()
        # Add all columns to dict
        storm_entries.append({
            "storm_id": storm_id,
            "storm_name": storm_name,
            "year": year,
            "date": date,
            "time": time,
            "record_identifier": record_identifier,
            "system_status": system_status,
            "lat": lat,
            "lon": lon,
            "max_sustained_wind": max_sustained_wind
        })
    i += n_entries + 1

storms_df = pd.DataFrame(storm_entries)

storms_df["datetime"] = pd.to_datetime(storms_df["date"] + " " + storms_df["time"], format="%Y%m%d %H%M")
storms_df = storms_df.drop(columns=["year", "date", "time", "record_identifier"])
storms_df['lat'] = storms_df['lat'].apply(clean_lat)
storms_df['lon'] = storms_df['lon'].apply(clean_lon)
storms_df["system_status_desc"] = storms_df["system_status"].map(system_status_mapping)

storms_df.to_csv("hurdat2_storm_data_1851_2025.csv", index=False)